# 개념 4. 확장 칼만 필터 (EKF)

**목표:** 움직임이 직선 관계가 아닐 때(비선형) 칼만 필터를 쓰는 방법을 이해한다.

예시: 평면 위를 일정한 속력으로 달리다가 **원을 그리며 도는** 장난감 자동차. 1초마다 위치만 잰다(±0.5m). 도는 속도는 재지 않는다.

순서
1. 도는 물체의 모델
2. 왜 칼만 필터로는 안 되나
3. 확장 칼만 필터: 기울기(야코비안)로 근사하기
4. 재지 않은 도는 속도 추정하기
5. 곡선으로 외삽하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"   # 한글 글꼴 (macOS)
plt.rcParams["axes.unicode_minus"] = False

## 1. 도는 물체의 모델

상태를 5개로 둔다: $\mathbf{x} = [x,\ y,\ v,\ \psi,\ \omega]$

| 기호 | 뜻 |
|---|---|
| $x, y$ | 위치 (m) |
| $v$ | 속력 (m/s) |
| $\psi$ | 진행 방향 (라디안, $x$축에서 반시계) |
| $\omega$ | 방향이 도는 속도 (라디안/초) |

*"속력과 도는 속도가 일정하다"*고 보면 원호를 그린다. $dt$초 뒤:

$$\psi' = \psi + \omega\,dt$$
$$x' = x + \frac{v}{\omega}(\sin\psi' - \sin\psi),\qquad y' = y + \frac{v}{\omega}(\cos\psi - \cos\psi')$$

$\omega = 0$이면 0으로 나누게 되므로 직선 식 $x' = x + v\cos\psi\,dt$를 쓴다.

In [ ]:
def move(s, dt):
    """등속 선회 모델로 상태 s = [x, y, v, psi, omega]를 dt 초 옮긴다."""
    x, y, v, psi, w = s
    if abs(w) < 1e-6:
        return np.array([x + v * np.cos(psi) * dt, y + v * np.sin(psi) * dt, v, psi, w])
    psi2 = psi + w * dt
    return np.array([x + v / w * (np.sin(psi2) - np.sin(psi)),
                     y + v / w * (np.cos(psi) - np.cos(psi2)), v, psi2, w])

fig, ax = plt.subplots(figsize=(5, 5))
for w in [0, 0.1, 0.3, 0.6]:
    path = np.array([move([0, 0, 2.0, 0, w], s)[:2] for s in np.linspace(0, 10, 50)])
    ax.plot(path[:, 0], path[:, 1], label=f"ω = {w} rad/s")
ax.set_aspect("equal"); ax.grid(alpha=.3); ax.legend(); ax.set_title("2 m/s로 10초, 도는 속도별 경로"); plt.show()

## 2. 왜 칼만 필터로는 안 되나

`move`에는 $\sin$, $\cos$, $v/\omega$가 들어 있다. 칼만 필터는 $\mathbf{x}^- = F\mathbf{x}$(행렬 곱)만 다루므로 이 식을 쓸 수 없다.

## 3. 확장 칼만 필터: 기울기로 근사하기

아이디어는 하나다.

- **상태**는 원래 식 그대로 옮긴다: $\mathbf{x}^- = f(\mathbf{x})$
- **불확실성**만 지금 지점의 기울기(**야코비안** $J$)로 직선처럼 근사해서 옮긴다: $P^- = JPJ^\top + Q$

곡선도 아주 가까이서 보면 직선처럼 보인다. 그 직선의 기울기가 $J$다.

$J$의 칸 $(i, j)$는 *"상태 $j$를 조금 바꾸면 결과 $i$가 얼마나 바뀌나"*다. 손으로 미분해도 되지만, 여기서는 **조금 흔들어서** 잰다.

In [ ]:
def jacobian(s, dt, eps=1e-6):
    """move 의 야코비안: 상태를 하나씩 eps 만큼 흔들어 결과가 바뀐 비율."""
    J = np.zeros((5, 5))
    base = move(s, dt)
    for j in range(5):
        d = np.zeros(5); d[j] = eps
        J[:, j] = (move(s + d, dt) - base) / eps
    return J

np.set_printoptions(precision=2, suppress=True)
print(jacobian(np.array([0, 0, 2.0, 0, 0.3]), 1.0))

마지막 열을 보면 $\omega$가 조금 바뀔 때 위치 $x, y$가 바뀐다. 거꾸로 말하면, **위치가 예측에서 옆으로 어긋나면 그걸 보고 $\omega$를 고칠 수 있다.** 이렇게 재지 않은 도는 속도가 추정된다.

보정 단계는 칼만 필터와 똑같다.

## 4. 재지 않은 도는 속도 추정하기

장난감 자동차는 2 m/s로 가다가 10~25초에 0.3 rad/s로 돌고, 다시 곧게 간다.

In [ ]:
dt = 1.0
t = np.arange(0, 41, dt)
rate = lambda s: 0.3 if 10 <= s < 25 else 0.0

truth, s = [], np.array([0, 0, 2.0, 0, 0])
for tk in t:
    s[4] = rate(tk); truth.append(s.copy()); s = move(s, dt)
truth = np.array(truth)
rng = np.random.default_rng(0)
z = truth[:, :2] + rng.normal(0, 0.5, (len(t), 2))      # 위치만 잰다

H = np.zeros((2, 5)); H[0, 0] = H[1, 1] = 1
R = np.eye(2) * 0.5**2
Q = np.diag([0, 0, 0.001, 0, 0.03]) * dt

x = np.array([z[0, 0], z[0, 1], 1.0, 0.0, 0.0])
P = np.diag([0.25, 0.25, 4, 1, 0.5])
est = []
for k in range(len(t)):
    if k > 0:                                            # 예측: 상태는 원래 식, 불확실성은 J 로
        J = jacobian(x, dt)
        x = move(x, dt)
        P = J @ P @ J.T + Q
    K = P @ H.T @ np.linalg.inv(H @ P @ H.T + R)         # 보정: 칼만 필터와 같다
    x = x + K @ (z[k] - H @ x)
    P = (np.eye(5) - K @ H) @ P
    est.append(x.copy())
est = np.array(est)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(truth[:, 0], truth[:, 1], color="lightgray", lw=4, label="실제")
axes[0].plot(z[:, 0], z[:, 1], "o", color="k", ms=3, label="측정 위치")
axes[0].plot(est[:, 0], est[:, 1], "-", label="EKF 추정")
axes[0].set_aspect("equal"); axes[0].grid(alpha=.3); axes[0].legend()
axes[1].plot(t, truth[:, 4], "k:", label="실제 도는 속도")
axes[1].plot(t, est[:, 4], "o-", ms=3, label="EKF 추정 (재지 않음!)")
axes[1].set_xlabel("초"); axes[1].set_ylabel("rad/s"); axes[1].grid(alpha=.3); axes[1].legend(); plt.show()

돌기 시작하고 몇 초 지나서야 알아챈다. 위치가 예측에서 어긋나는 걸 봐야 고칠 수 있기 때문이다.

## 5. 곡선으로 외삽하기

18초(도는 중)의 추정 상태에서 5초 앞을 외삽해 보자.
- 선형 외삽: 지금 방향으로 곧게
- EKF 상태로 외삽: `move`로 원호를 따라

In [ ]:
k = 18
ahead = np.linspace(0, 5, 30)
straight = np.array([[est[k, 0] + est[k, 2] * np.cos(est[k, 3]) * a,
                      est[k, 1] + est[k, 2] * np.sin(est[k, 3]) * a] for a in ahead])
curved = np.array([move(est[k], a)[:2] for a in ahead])
real = np.array([truth[min(k + int(a), len(t) - 1), :2] for a in range(6)])

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(truth[:, 0], truth[:, 1], color="lightgray", lw=4, label="실제")
ax.plot(straight[:, 0], straight[:, 1], "--", color="C3", label="선형 외삽 (곧게)")
ax.plot(curved[:, 0], curved[:, 1], "-", color="C2", lw=2, label="EKF 상태로 외삽 (곡선)")
ax.plot(real[:, 0], real[:, 1], "o", color="k", ms=4, label="실제 5초 동안")
ax.set_aspect("equal"); ax.grid(alpha=.3); ax.legend(); plt.show()

print(f"5초 뒤 오차: 곧게 {np.hypot(*(straight[-1] - real[-1])):.1f} m, 곡선 {np.hypot(*(curved[-1] - real[-1])):.1f} m")

## 직접 해 보기

1. `Q`의 마지막 값 `0.03`을 `0.3`으로 키우면 도는 속도를 더 빨리 알아채나? 직진할 때는 어떻게 되나?
2. 측정 잡음 `0.5`를 `2.0`으로 키우면 도는 속도 추정은 어떻게 되나?

In [ ]:
# 여기서 직접 바꿔 보세요


## 정리

| | 칼만 필터 | 확장 칼만 필터 |
|---|---|---|
| 움직임 식 | 직선 관계 ($F\mathbf{x}$) | 비선형 ($f(\mathbf{x})$, sin·cos 등) |
| 상태 예측 | $F\mathbf{x}$ | $f(\mathbf{x})$ 그대로 |
| 불확실성 예측 | $FPF^\top + Q$ | $JPJ^\top + Q$ ($J$: 지금 지점의 기울기) |
| 외삽 모양 | 직선 | 모델대로 (여기서는 곡선) |

개념은 여기까지다. **다음부터는 해양 응용**: 이 네 가지를 AIS 보고를 받는 배 위치에 쓴다.